In [1]:
# 1
import math
from ortools.sat.python import cp_model


class Robot:
    def __init__(self, size, obstacles, start, end):
        self.size = size
        self.obstacles = set(obstacles)
        self.start = start
        self.end = end
        self.model = cp_model.CpModel()

    def solve(self):
        max_steps = self.size * self.size

        # Variables
        px = [
            self.model.NewIntVar(0, self.size - 1, f"x_{t}") for t in range(max_steps)
        ]
        py = [
            self.model.NewIntVar(0, self.size - 1, f"y_{t}") for t in range(max_steps)
        ]

        # State Constraints
        self.model.Add(px[0] == self.start[0])
        self.model.Add(py[0] == self.start[1])

        # Movement Logic
        is_moving = []
        for t in range(max_steps - 1):
            dx = self.model.NewIntVar(-1, 1, f"dx_{t}")
            dy = self.model.NewIntVar(-1, 1, f"dy_{t}")
            abs_dx = self.model.NewIntVar(0, 1, f"abs_dx_{t}")
            abs_dy = self.model.NewIntVar(0, 1, f"abs_dy_{t}")

            self.model.Add(dx == px[t + 1] - px[t])
            self.model.Add(dy == py[t + 1] - py[t])
            self.model.AddAbsEquality(abs_dx, dx)
            self.model.AddAbsEquality(abs_dy, dy)

            # Diagonal Move: |dx| == |dy|
            self.model.Add(abs_dx == abs_dy)

            # move_occurred is 1 if diagonal, 0 if staying
            move_occurred = self.model.NewBoolVar(f"move_{t}")
            self.model.Add(abs_dx == 1).OnlyEnforceIf(move_occurred)
            self.model.Add(abs_dx == 0).OnlyEnforceIf(move_occurred.Not())
            is_moving.append(move_occurred)

            if t > 0:
                self.model.AddImplication(is_moving[t].Not(), is_moving[t - 1].Not())
                # Better: If t is not moving, then t+1 cannot move
                self.model.Add(is_moving[t] <= is_moving[t - 1])

        # Obstacles & Goal
        for t in range(max_steps):
            for ox, oy in self.obstacles:
                self.model.AddForbiddenAssignments([px[t], py[t]], [(ox, oy)])

        # Reaching the goal at some point
        reaches_goal = [self.model.NewBoolVar(f"goal_{t}") for t in range(max_steps)]
        for t in range(max_steps):
            at_x = self.model.NewBoolVar(f"at_x_{t}")
            at_y = self.model.NewBoolVar(f"at_y_{t}")
            self.model.Add(px[t] == self.end[0]).OnlyEnforceIf(at_x)
            self.model.Add(py[t] == self.end[1]).OnlyEnforceIf(at_y)
            self.model.AddBoolAnd([at_x, at_y]).OnlyEnforceIf(reaches_goal[t])

        # Must reach goal at least once
        self.model.AddBoolOr(reaches_goal)

        # Minimize total diagonal moves
        self.model.Minimize(sum(is_moving))

        solver = cp_model.CpSolver()
        status = solver.Solve(self.model)

        if status in [cp_model.OPTIMAL, cp_model.FEASIBLE]:
            raw_path = []
            for t in range(max_steps):
                coord = (solver.Value(px[t]), solver.Value(py[t]))
                raw_path.append(coord)
                if coord == self.end:
                    break

            final_path = [raw_path[0]]
            for pos in raw_path[1:]:
                if pos != final_path[-1]:
                    final_path.append(pos)

            actual_steps = len(final_path) - 1
            cost = actual_steps * math.sqrt(2)

            print(f"Optimal Path: {final_path}")
            print(f"Total Steps: {actual_steps}")
            print(f"Pythagorean Cost: {cost:.4f}")
        else:
            print("No path found.")


obstacles = [(2, 2)]
robot = Robot(size=5, obstacles=obstacles, start=(1, 1), end=(4, 4))
robot.solve()

Optimal Path: [(1, 1), (0, 2), (1, 3), (0, 2), (1, 1), (0, 2), (1, 3), (2, 4), (3, 3), (4, 4)]
Total Steps: 9
Pythagorean Cost: 12.7279


In [2]:
# 2
from ortools.sat.python import cp_model
from collections import deque


GRID = [
    [0, 1, 1, 0, 0, 0, 1, 1],
    [0, 1, 1, 1, 0, 0, 1, 0],
    [0, 0, 1, 1, 1, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 0],
    [1, 1, 0, 0, 0, 1, 1, 1],
    [1, 1, 1, 0, 0, 1, 0, 1],
    [0, 1, 0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 0, 0, 0],
]

ROWS = len(GRID)
COLS = len(GRID[0])
DIRECTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # up, down, left, right


def build_csp_model(grid):
    model = cp_model.CpModel()

    # Create one BoolVar per cell
    cell_vars = [
        [model.new_bool_var(f"cell_{r}_{c}") for c in range(COLS)] for r in range(ROWS)
    ]

    # Constraint: each variable must equal its observed satellite value
    for r in range(ROWS):
        for c in range(COLS):
            if grid[r][c] == 1:
                model.add(cell_vars[r][c] == 1)
            else:
                model.add(cell_vars[r][c] == 0)

    # Solve
    solver = cp_model.CpSolver()
    status = solver.solve(model)

    return model, solver, cell_vars, status


def find_all_islands(grid):
    visited = [[False] * COLS for _ in range(ROWS)]
    islands = []

    for r in range(ROWS):
        for c in range(COLS):
            if grid[r][c] == 1 and not visited[r][c]:
                # BFS to collect all connected land cells
                island = set()
                queue = deque([(r, c)])
                visited[r][c] = True
                while queue:
                    cr, cc = queue.popleft()
                    island.add((cr, cc))
                    for dr, dc in DIRECTIONS:
                        nr, nc = cr + dr, cc + dc
                        if (
                            0 <= nr < ROWS
                            and 0 <= nc < COLS
                            and grid[nr][nc] == 1
                            and not visited[nr][nc]
                        ):
                            visited[nr][nc] = True
                            queue.append((nr, nc))
                islands.append(island)

    return islands


def compute_perimeter(island, solver, cell_vars):
    perimeter = 0
    boundary_edges = []

    for r, c in island:
        for dr, dc in DIRECTIONS:
            nr, nc = r + dr, c + dc
            # Edge is a boundary if neighbor is out-of-bounds OR is water (CSP var = 0)
            if not (0 <= nr < ROWS and 0 <= nc < COLS):
                perimeter += 1
                boundary_edges.append(((r, c), (nr, nc), "grid-border"))
            elif solver.value(cell_vars[nr][nc]) == 0:
                perimeter += 1
                boundary_edges.append(((r, c), (nr, nc), "water-border"))

    return perimeter, boundary_edges


def print_grid_highlighted(grid, highlight_cells, label="Grid"):
    print(f"  {label}\n")
    for r in range(ROWS):
        row_str = "  "
        for c in range(COLS):
            if (r, c) in highlight_cells:
                row_str += f"[{grid[r][c]}]"
            else:
                row_str += f" {grid[r][c]} "
        print(row_str)


def print_section(title):
    print(f"  {title}\n")


#  Main
def main():
    print_section("SATELLITE ISLAND EROSION TRACKER")
    print(f"  Grid dimensions : {ROWS} × {COLS}")
    print(f"  Total cells     : {ROWS * COLS}")
    land_count = sum(GRID[r][c] for r in range(ROWS) for c in range(COLS))
    print(f"  Land cells      : {land_count}")
    print(f"  Water cells     : {ROWS * COLS - land_count}")

    # Step 1: Build & solve CSP model
    print_section("\nCSP Model — Binary Variable Assignment")
    model, solver, cell_vars, status = build_csp_model(GRID)

    status_name = solver.status_name(status)
    print(f"  Solver status   : {status_name}")
    print(f"  Variables       : {ROWS * COLS} BoolVars")
    print(f"  Constraints     : {ROWS * COLS} equality constraints")

    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        print("  CSP model is infeasible. Exiting.")
        return

    print("\n  CSP Variable Values (matches satellite grid):")
    for r in range(ROWS):
        row_vals = "  " + " ".join(
            str(solver.value(cell_vars[r][c])) for c in range(COLS)
        )
        print(row_vals)

    # Step 2: Identify all islands
    print_section("\nIsland Detection via BFS")
    islands = find_all_islands(GRID)
    print(f"  Islands found   : {len(islands)}")
    for i, island in enumerate(islands):
        print(f"    Island {i + 1:>2}     : {len(island)} cells  →  {sorted(island)}")

    largest_island = max(islands, key=len)
    print(f"\n  ★ Largest island: {len(largest_island)} cells")

    print_grid_highlighted(GRID, largest_island, label="Largest Island [highlighted]")

    # Step 3: Compute perimeter using CSP values
    print_section("\nPerimeter Computation via CSP Boundary Edges")
    perimeter, boundary_edges = compute_perimeter(largest_island, solver, cell_vars)

    grid_border_edges = [e for e in boundary_edges if e[2] == "grid-border"]
    water_border_edges = [e for e in boundary_edges if e[2] == "water-border"]

    print(f"  Total perimeter edges : {perimeter}")
    print(f"    Grid boundary    : {len(grid_border_edges)}")
    print(f"    Land↔Water edges : {len(water_border_edges)}")

    print(f"  Largest island size : {len(largest_island)} cells")
    print(f"  Perimeter           : {perimeter} units")


if __name__ == "__main__":
    main()

  SATELLITE ISLAND EROSION TRACKER

  Grid dimensions : 8 × 8
  Total cells     : 64
  Land cells      : 28
  Water cells     : 36
  
CSP Model — Binary Variable Assignment

  Solver status   : OPTIMAL
  Variables       : 64 BoolVars
  Constraints     : 64 equality constraints

  CSP Variable Values (matches satellite grid):
  0 1 1 0 0 0 1 1
  0 1 1 1 0 0 1 0
  0 0 1 1 1 0 0 0
  0 0 0 1 0 0 0 0
  1 1 0 0 0 1 1 1
  1 1 1 0 0 1 0 1
  0 1 0 0 0 1 1 1
  0 0 0 1 1 0 0 0
  
Island Detection via BFS

  Islands found   : 5
    Island  1     : 9 cells  →  [(0, 1), (0, 2), (1, 1), (1, 2), (1, 3), (2, 2), (2, 3), (2, 4), (3, 3)]
    Island  2     : 3 cells  →  [(0, 6), (0, 7), (1, 6)]
    Island  3     : 6 cells  →  [(4, 0), (4, 1), (5, 0), (5, 1), (5, 2), (6, 1)]
    Island  4     : 8 cells  →  [(4, 5), (4, 6), (4, 7), (5, 5), (5, 7), (6, 5), (6, 6), (6, 7)]
    Island  5     : 2 cells  →  [(7, 3), (7, 4)]

  ★ Largest island: 9 cells
  Largest Island [highlighted]

   0 [1][1] 0  0  0  1  1 
 

In [3]:
# 3
from ortools.sat.python import cp_model

CITIES = {
    0: ("New York", 0, 0),
    1: ("Los Angeles", 28, 8),
    2: ("Chicago", 13, 5),
    3: ("Houston", 18, 20),
    4: ("Phoenix", 22, 14),
    5: ("Philadelphia", 3, 2),
    6: ("San Antonio", 19, 22),
    7: ("San Diego", 26, 10),
    8: ("Dallas", 17, 18),
    9: ("San Jose", 25, 5),
}

NUM_CITIES = len(CITIES)


def build_distance_matrix():
    # Scaled by 100 — CP-SAT requires integer coefficients
    dist = {}
    for i in range(NUM_CITIES):
        for j in range(NUM_CITIES):
            x1, y1 = CITIES[i][1], CITIES[i][2]
            x2, y2 = CITIES[j][1], CITIES[j][2]
            dist[(i, j)] = int(((x2 - x1) ** 2 + (y2 - y1) ** 2) ** 0.5 * 100)
    return dist


def print_overview():
    print("  TRAVELLING SALESMAN PROBLEM — CP-SAT Formulation")
    print()
    print(f"  Cities     : {NUM_CITIES}")
    print(f"  Start/End  : {CITIES[0][0]} (index 0)")
    print("\n  CSP Definition")
    print("=" * 55)
    print("  Variables   : position[0..9] — city at each step")
    print("  Domains     : {0, 1, ..., 9} per variable")
    print("  Constraints :")
    print("    1. AllDifferent  — no city repeated")
    print("    2. position[0]   — fixed to start city (0)")
    print("    3. Minimize      — total Euclidean distance")
    print("=" * 55)


def solve_tsp(dist):
    model = cp_model.CpModel()

    # Variables
    # position[i] = which city is visited at step i
    position = [
        model.new_int_var(0, NUM_CITIES - 1, f"pos_{i}") for i in range(NUM_CITIES)
    ]

    # arc[i][j] = 1 if city j is visited directly after city i
    arc = {
        (i, j): model.new_bool_var(f"arc_{i}_{j}")
        for i in range(NUM_CITIES)
        for j in range(NUM_CITIES)
        if i != j
    }

    # Constraints:
    # Each city visited exactly once
    model.add_all_different(position)

    # Fix start to city 0
    model.add(position[0] == 0)

    # Each city has exactly one successor and one predecessor
    for i in range(NUM_CITIES):
        model.add(sum(arc[(i, j)] for j in range(NUM_CITIES) if i != j) == 1)
        model.add(sum(arc[(j, i)] for j in range(NUM_CITIES) if i != j) == 1)

    # Circuit constraint — ensures a single closed tour (no subtours)
    model.add_circuit(
        [
            (i, j, arc[(i, j)])
            for i in range(NUM_CITIES)
            for j in range(NUM_CITIES)
            if i != j
        ]
    )

    # Objective
    total_distance = sum(dist[(i, j)] * arc[(i, j)] for i, j in arc)
    model.minimize(total_distance)

    # Solve
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 10.0
    status = solver.solve(model)

    return solver, status, arc


def print_solution(solver, status, arc, dist):
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        print("  No solution found.")
        return

    # Reconstruct tour from arc variables
    next_city = {i: j for (i, j) in arc if solver.value(arc[(i, j)]) == 1}

    tour = [0]
    while len(tour) < NUM_CITIES:
        tour.append(next_city[tour[-1]])

    total = 0
    print(f"\n  Status : {solver.status_name(status)}")
    print("\n  OPTIMAL TOUR")
    print("  -------------------------------------------------")
    for step in range(NUM_CITIES):
        current = tour[step]
        nxt = tour[(step + 1) % NUM_CITIES]
        leg = dist[(current, nxt)] / 100
        total += leg
        print(
            f"  Step {step + 1:>2}: {CITIES[current][0]:<15} → {CITIES[nxt][0]:<15}  dist: {leg:.2f}"
        )
    print("  -------------------------------------------------")
    print()
    print(f"  Total distance : {total:.2f} units")
    route_names = " → ".join(CITIES[c][0] for c in tour) + f" → {CITIES[tour[0]][0]}"
    print(f"  Route : {route_names}")


if __name__ == "__main__":
    print_overview()
    dist = build_distance_matrix()
    solver, status, arc = solve_tsp(dist)
    print_solution(solver, status, arc, dist)

  TRAVELLING SALESMAN PROBLEM — CP-SAT Formulation

  Cities     : 10
  Start/End  : New York (index 0)

  CSP Definition
  Variables   : position[0..9] — city at each step
  Domains     : {0, 1, ..., 9} per variable
  Constraints :
    1. AllDifferent  — no city repeated
    2. position[0]   — fixed to start city (0)
    3. Minimize      — total Euclidean distance

  Status : OPTIMAL

  OPTIMAL TOUR
  -------------------------------------------------
  Step  1: New York        → Chicago          dist: 13.92
  Step  2: Chicago         → San Jose         dist: 12.00
  Step  3: San Jose        → Los Angeles      dist: 4.24
  Step  4: Los Angeles     → San Diego        dist: 2.82
  Step  5: San Diego       → Phoenix          dist: 5.65
  Step  6: Phoenix         → San Antonio      dist: 8.54
  Step  7: San Antonio     → Houston          dist: 2.23
  Step  8: Houston         → Dallas           dist: 2.23
  Step  9: Dallas          → Philadelphia     dist: 21.26
  Step 10: Philadelphia    →